In [ ]:
# %pip install numpy pandas scikit-learn keras tensorflow matplotlib seaborn scapy xgboost

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
data = pd.read_csv('Thursday.csv')
df = data.copy()


# df = pd.read_csv('Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv')

In [ ]:
df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
label_encoder = LabelEncoder() 
flag = "2D"
if flag ==  "MD":
    label_encoder = LabelEncoder()
    df['Label'] = label_encoder.fit_transform(df['Label'])
    print("Etykiety odpowiadające wartościom liczbowym:")
    for i in range(len(label_encoder.classes_)):
        print(f"{i}: {label_encoder.classes_[i]}")
elif flag == "2D":
    df['Label'] = df['Label'].map(lambda x: 0 if x == "BENIGN" else 1)


print("Unikalne wartości w kolumnie 'Label':")
print(df['Label'].unique())
print(df['Label'].value_counts()) 

In [ ]:
df.columns = df.columns.str.replace(' ', '_')
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

In [ ]:
X = df.drop('Label', axis=1).values
y = df['Label'].values

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X = scaler.fit_transform(X)

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Obliczanie wag klasowych
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights))

X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

In [ ]:
for each in y_train:
    if each != 0 and each != 1:
        print(each)

In [ ]:
def build_model_2d():
    model = Sequential()

    model.add(LSTM(256, activation='relu', return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
    model.add(Dropout(0.2))  

    model.add(LSTM(128, activation='relu', return_sequences=True))
    model.add(Dropout(0.2))

    model.add(LSTM(64, activation='relu'))
    model.add(Dropout(0.2))

    model.add(Dense(1, activation='sigmoid'))  # 1 klasa dla klasyfikacji binarnej
    return model

In [ ]:
def build_model_md():
    model = Sequential()

    model.add(LSTM(256, activation='relu', return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
    model.add(Dropout(0.2))  # Dropout dla regularizacji

    model.add(LSTM(128, activation='relu', return_sequences=True))
    model.add(Dropout(0.2))

    model.add(LSTM(64, activation='relu'))
    model.add(Dropout(0.2))

    model.add(Dense(2, activation='softmax'))  # 2 klasy: benign i attack
    return model

In [ ]:
optimizer = Adam(learning_rate=0.001)

model = build_model_2d()
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['precision', 'recall'])

# model = build_model_md()
# Kompilacja modelu z optymalizatorem Adam
# model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=10)

# model.fit(X_train, y_train, epochs=5, batch_size=120, validation_split=0.2, callbacks=[early_stopping])
model.fit(X_train, y_train, epochs=5, batch_size=32, class_weight=class_weights, validation_data=(X_test, y_test))

In [ ]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

In [ ]:
cm = confusion_matrix(y_test, y_pred_classes)

In [ ]:
# TP, TN, FP, FN
TN, FP, FN, TP = cm.ravel()

print(f'True Positives (TP): {TP}')
print(f'True Negatives (TN): {TN}')
print(f'False Positives (FP): {FP}')
print(f'False Negatives (FN): {FN}')

print(classification_report(y_test, y_pred_classes, target_names=['BENIGN', 'ATTACK']))
# print(classification_report(y_test, y_pred_classes, target_names=label_encoder.classes_))

True Positives (TP): 0  
True Negatives (TN): 91292  
False Positives (FP): 0  
False Negatives (FN): 434  
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     91292
      ATTACK       0.00      0.00      0.00       434

    accuracy                           1.00     91726
   macro avg       0.50      0.50      0.50     91726